In [3]:
# ════════════════════════════════════════════════════════════
#  Combined Model Evaluation: input → plan [SQL] sql
#  Evaluates Plan F1 and SQL F1 on 150 dev samples
# ════════════════════════════════════════════════════════════
!pip install -q transformers==4.40.0 peft==0.10.0 accelerate==0.29.3 sentence-transformers==2.2.2

import json, zipfile, os, re, gc, time
from collections import Counter
import torch
from transformers import T5ForConditionalGeneration, AutoTokenizer
from peft import PeftModel

# ── Config ────────────────────────────────────────────────────
ADAPTER_ZIP  = "./lora_adapter (1) (2).zip"
ADAPTER_DIR  = "./lora_adapter_combined"
DEV_PATH     = "./dev_final.json"              # <-- upload dev_final.json
OUTPUT_PATH  = "./combined_eval_results.json"
SQL_SEP      = "[SQL]"
MAX_INPUT    = 512
MAX_TARGET   = 512
N_SAMPLES    = 150

# ── 1. Unzip adapter ──────────────────────────────────────────
if not os.path.exists(ADAPTER_DIR):
    with zipfile.ZipFile(ADAPTER_ZIP, 'r') as z:
        z.extractall(ADAPTER_DIR)
    print("Extracted to:", ADAPTER_DIR)

# ── 2. Load model ─────────────────────────────────────────────
MODEL_NAME = "google/flan-t5-xl"

dtype = torch.bfloat16 if torch.cuda.is_available() and \
        torch.cuda.get_device_capability()[0] >= 8 else torch.float16

print(f"Loading base model ({dtype})...")
base_model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype = dtype,
    device_map  = "auto",
)

print("Loading combined LoRA...")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR, is_trainable=False)
model.eval()
model.config.use_cache = True

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
print("Model ready!")

# ── 3. Load dev data ──────────────────────────────────────────
with open(DEV_PATH, encoding="utf-8") as f:
    dev_data = json.load(f)

dev_samples = dev_data[:N_SAMPLES]
print(f"Loaded {len(dev_samples)} samples")

# ── 4. Inference ──────────────────────────────────────────────
def generate(input_text):
    inp = tokenizer(
        input_text,
        return_tensors = "pt",
        max_length     = MAX_INPUT,
        truncation     = True,
    )
    inp = {k: v.to(model.device) for k, v in inp.items()}
    with torch.no_grad():
        out = model.generate(
            **inp,
            max_new_tokens = MAX_TARGET,
            num_beams      = 4,
            early_stopping = True,
            length_penalty = 1.0,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

def split_output(text):
    """Split 'plan [SQL] sql' into (plan, sql)"""
    if SQL_SEP in text:
        plan_part, sql_part = text.split(SQL_SEP, 1)
        return plan_part.strip(), sql_part.strip()
    return "", text.strip()   # no separator found — treat all as sql

# ── 5. Generate predictions ───────────────────────────────────
results = []
start = time.time()

for i, sample in enumerate(dev_samples):
    pred_text = generate(sample["input"])
    pred_plan, pred_sql = split_output(pred_text)

    results.append({
        "idx"       : i,
        "input"     : sample["input"],
        "gold_plan" : sample["output"],     # ground truth plan
        "gold_sql"  : sample["sql"],        # ground truth sql
        "pred_plan" : pred_plan,
        "pred_sql"  : pred_sql,
        "pred_raw"  : pred_text,            # full raw output before split
    })

    if (i + 1) % 25 == 0:
        elapsed = time.time() - start
        eta = elapsed / (i + 1) * (N_SAMPLES - i - 1)
        print(f"  [{i+1}/{N_SAMPLES}]  elapsed: {elapsed/60:.1f} min  ETA: {eta/60:.1f} min")

# ── 6. Token F1 ───────────────────────────────────────────────
def tokenize(text):
    return re.findall(r'\w+', text.lower())

def token_f1(pred, gold):
    pred_tokens = tokenize(pred)
    gold_tokens = tokenize(gold)
    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0
    pred_count  = Counter(pred_tokens)
    gold_count  = Counter(gold_tokens)
    overlap     = sum((pred_count & gold_count).values())
    precision   = overlap / len(pred_tokens)
    recall      = overlap / len(gold_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

plan_f1_scores = []
sql_f1_scores  = []

for r in results:
    pf1 = token_f1(r["pred_plan"], r["gold_plan"])
    sf1 = token_f1(r["pred_sql"],  r["gold_sql"])
    plan_f1_scores.append(pf1)
    sql_f1_scores.append(sf1)
    r["plan_f1"] = round(pf1, 4)
    r["sql_f1"]  = round(sf1, 4)

avg_plan_f1 = sum(plan_f1_scores) / len(plan_f1_scores)
avg_sql_f1  = sum(sql_f1_scores)  / len(sql_f1_scores)

print("\n══════════════════════════════════════════")
print(f"   COMBINED MODEL EVALUATION (n={N_SAMPLES})")
print("══════════════════════════════════════════")
print(f"  Plan Token F1 : {avg_plan_f1*100:.2f}%")
print(f"  SQL  Token F1 : {avg_sql_f1*100:.2f}%")
print("══════════════════════════════════════════")

# ── 7. Save ───────────────────────────────────────────────────
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"\nSaved → {OUTPUT_PATH}")

# ── 8. Sample predictions ─────────────────────────────────────
print("\n── Sample predictions ──")
for i in [0, 10, 50]:
    if i < len(results):
        r = results[i]
        print(f"\n[{i}] Plan F1={r['plan_f1']:.3f}  SQL F1={r['sql_f1']:.3f}")
        print(f"  INPUT     : {r['input'][:80]}...")
        print(f"  GOLD PLAN : {r['gold_plan']}")
        print(f"  PRED PLAN : {r['pred_plan']}")
        print(f"  GOLD SQL  : {r['gold_sql']}")
        print(f"  PRED SQL  : {r['pred_sql']}")

Loading base model (torch.float16)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.45G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loading combined LoRA...
Model ready!
Loaded 150 samples
  [25/150]  elapsed: 3.1 min  ETA: 15.7 min
  [50/150]  elapsed: 8.5 min  ETA: 17.1 min
  [75/150]  elapsed: 15.2 min  ETA: 15.2 min
  [100/150]  elapsed: 20.5 min  ETA: 10.2 min
  [125/150]  elapsed: 26.0 min  ETA: 5.2 min
  [150/150]  elapsed: 30.5 min  ETA: 0.0 min

══════════════════════════════════════════
   COMBINED MODEL EVALUATION (n=150)
══════════════════════════════════════════
  Plan Token F1 : 93.93%
  SQL  Token F1 : 94.02%
══════════════════════════════════════════

Saved → ./combined_eval_results.json

── Sample predictions ──

[0] Plan F1=1.000  SQL F1=1.000
  INPUT     : question: How many singers do we have? | schema: singer ( Singer_ID [PK] ) | for...
  GOLD PLAN : step1: SCAN | table: singer || step2: AGGREGATE | Scalar aggregate (no GROUP BY)  ->  compute COUNT(*) || step3: PROJECT | columns: COUNT(*)
  PRED PLAN : step1: SCAN | table: singer || step2: AGGREGATE | Scalar aggregate (no GROUP BY) -> compute C

In [1]:
from google.colab import files
uploaded = files.upload()



Saving dev_final.json to dev_final (1).json


In [2]:
from google.colab import files
uploaded = files.upload()

Saving lora_adapter (1) (2).zip to lora_adapter (1) (2) (1).zip


In [6]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print("GPU count:", torch.cuda.device_count())
else:
    print("Running on CPU")

CUDA available: True
Device: Tesla T4
CUDA version: 12.8
GPU count: 1
